In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

from cv_caseyanderson.content import load_cv

project_root = Path.cwd().parent
cv_path = project_root / "data" / "cv.json"

cv = load_cv(cv_path)

print("project root:", project_root)
print("data path:", cv_path)
print("top-level keys:", cv.keys())

In [ ]:
import bs4

print(bs4.__version__)

In [ ]:
source_html_path = project_root / "index.html"
source_html = source_html_path.read_text(
    encoding="utf-8"
)

print("characters:", len(source_html))

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(
    source_html,
    "html.parser",
)

print("html found:", soup.html is not None)
print("body found:", soup.body is not None)

In [ ]:
print("before search")

source_sections = soup.body.find_all(
    "div",
    recursive=False,
)

print("after search")
print("sections found:", len(source_sections))

In [ ]:
for section in source_sections:
    heading = section.find("h2")
    print(heading.get_text(strip=True))

In [ ]:
for section in source_sections:
    section_title = section.find(
        "h2"
    ).get_text(strip=True)

    lists = section.find_all(
        "ul",
        recursive=False,
    )

    print()
    print(section_title)

    for item_list in lists:
        group_heading = item_list.find(
            "h3",
            recursive=False,
        )

        group_title = (
            group_heading.get_text(strip=True)
            if group_heading is not None
            else None
        )

        entries = item_list.find_all(
            "li",
            recursive=False,
        )

        print(
            "group:",
            group_title,
            "entries:",
            len(entries),
        )

In [ ]:
from collections import Counter


for section in source_sections:
    section_title = section.find(
        "h2"
    ).get_text(strip=True)

    entry_types = Counter()

    for entry in section.find_all("li"):
        boxed = entry.find(
            "span",
            class_="boxed",
            recursive=False,
        )

        link = entry.find(
            "a",
            recursive=False,
        )

        if boxed is not None:
            entry_types["boxed"] += 1
        elif link is not None:
            entry_types["link"] += 1
        else:
            entry_types["plain"] += 1

    print(section_title, dict(entry_types))

In [ ]:
for section in source_sections:
    section_title = section.find(
        "h2"
    ).get_text(strip=True)

    for entry in section.find_all("li"):
        primary = (
            entry.find(
                "span",
                class_="boxed",
                recursive=False,
            )
            or entry.find(
                "a",
                recursive=False,
            )
        )

        if primary is None:
            continue

        full_text = entry.get_text(
            " ",
            strip=True,
        )

        label = primary.get_text(
            " ",
            strip=True,
        )

        label_start = full_text.find(label)
        prefix = full_text[:label_start]

        if prefix:
            print(
                section_title,
                repr(prefix),
                repr(label),
            )

In [ ]:
def extract_entry(entry, section_title):
    full_text = entry.get_text(
        " ",
        strip=True,
    )

    boxed = entry.find(
        "span",
        class_="boxed",
        recursive=False,
    )

    link = entry.find(
        "a",
        recursive=False,
    )

    if boxed is not None:
        label = boxed.get_text(
            " ",
            strip=True,
        )
        label_start = full_text.find(label)

        result = {
            "type": "boxed",
            "prefix": full_text[:label_start],
            "label": label,
            "suffix": full_text[
                label_start + len(label):
            ],
        }

        if section_title == "Selected Discography":
            result["status"] = (
                "planned"
                if full_text.startswith("(")
                and full_text.endswith(")")
                else "published"
            )

        return result

    if link is not None:
        label = link.get_text(
            " ",
            strip=True,
        )
        label_start = full_text.find(label)

        return {
            "type": "link",
            "prefix": full_text[:label_start],
            "label": label,
            "url": link.get("href"),
            "suffix": full_text[
                label_start + len(label):
            ],
        }

    return {
        "type": "plain",
        "text": full_text,
    }

In [ ]:
entry_type_counts = {}
planned_entries = []

for section in source_sections:
    section_title = section.find("h2").get_text(strip=True)

    for entry in section.find_all("li"):
        extracted = extract_entry(
            entry,
            section_title,
        )

        entry_type = extracted["type"]
        entry_type_counts[entry_type] = (
            entry_type_counts.get(entry_type, 0) + 1
        )

        if extracted.get("status") == "planned":
            planned_entries.append(extracted)

print("entry types:", entry_type_counts)
print("planned entries:", planned_entries)

In [ ]:
reconstruction_failures = []

for section in source_sections:
    section_title = section.find("h2").get_text(strip=True)

    for entry in section.find_all("li"):
        source_text = entry.get_text(
            " ",
            strip=True,
        )

        extracted = extract_entry(
            entry,
            section_title,
        )

        if extracted["type"] == "plain":
            reconstructed_text = extracted["text"]
        else:
            reconstructed_text = "".join(
                [
                    extracted["prefix"],
                    extracted["label"],
                    extracted["suffix"],
                ]
            )

        if reconstructed_text != source_text:
            reconstruction_failures.append(
                {
                    "section": section_title,
                    "source": source_text,
                    "reconstructed": reconstructed_text,
                }
            )

print("entries tested:", sum(
    len(section.find_all("li"))
    for section in source_sections
))
print(
    "reconstruction failures:",
    len(reconstruction_failures),
)
print(reconstruction_failures)

In [ ]:
for section in source_sections:
    section_title = section.find(
        "h2",
        recursive=False,
    ).get_text(strip=True)

    structure = []

    for child in section.find_all(
        recursive=False,
    ):
        if child.name == "h2":
            structure.append("section title")
        elif child.name == "h3":
            structure.append(
                "group: "
                + child.get_text(strip=True)
            )
        elif child.name == "ul":
            structure.append(
                "entries: "
                + str(
                    len(
                        child.find_all(
                            "li",
                            recursive=False,
                        )
                    )
                )
            )
        else:
            structure.append(
                "other: " + child.name
            )

    print(section_title, structure)

In [ ]:
for section in source_sections:
    section_title = section.find(
        "h2",
        recursive=False,
    ).get_text(strip=True)

    group_headings = section.find_all("h3")

    if not group_headings:
        continue

    print(section_title)

    for heading in group_headings:
        print(
            {
                "title": heading.get_text(
                    " ",
                    strip=True,
                ),
                "parent": heading.parent.name,
                "parent_classes": heading.parent.get(
                    "class",
                ),
                "next_element": (
                    heading.find_next_sibling().name
                    if heading.find_next_sibling()
                    is not None
                    else None
                ),
            }
        )

In [ ]:
migrated_sections = []

for section in source_sections:
    section_heading = section.find(
        "h2",
        recursive=False,
    )

    groups = []

    for entry_list in section.find_all(
        "ul",
        recursive=False,
    ):
        group_heading = entry_list.find(
            "h3",
            recursive=False,
        )

        group_title = (
            group_heading.get_text(
                " ",
                strip=True,
            )
            if group_heading is not None
            else None
        )

        entries = [
            extract_entry(
                entry,
                section_heading.get_text(
                    " ",
                    strip=True,
                ),
            )
            for entry in entry_list.find_all(
                "li",
                recursive=False,
            )
        ]

        groups.append(
            {
                "title": group_title,
                "entries": entries,
            }
        )

    migrated_sections.append(
        {
            "id": section.get("id"),
            "title": section_heading.get_text(
                " ",
                strip=True,
            ),
            "groups": groups,
        }
    )

migrated_cv = {
    "sections": migrated_sections,
}

In [ ]:
print("sections:", len(migrated_cv["sections"]))
print(
    "groups:",
    sum(
        len(section["groups"])
        for section in migrated_cv["sections"]
    ),
)
print(
    "entries:",
    sum(
        len(group["entries"])
        for section in migrated_cv["sections"]
        for group in section["groups"]
    ),
)

for section in migrated_cv["sections"]:
    print(
        section["id"],
        section["title"],
        [
            (
                group["title"],
                len(group["entries"]),
            )
            for group in section["groups"]
        ],
    )

In [ ]:
for section in source_sections:
    heading = section.find(
        "h2",
        recursive=False,
    )

    print(
        {
            "title": heading.get_text(
                " ",
                strip=True,
            ),
            "section_attributes": dict(
                section.attrs
            ),
            "heading_attributes": dict(
                heading.attrs
            ),
        }
    )

In [ ]:
section_ids = {
    "Education": "education",
    "Professional Experience": "professional-experience",
    "Teaching": "teaching",
    "Guest Lectures & Artist Talks": "guest-lectures-artist-talks",
    "Selected Awards, Distinctions, and Events": "selected-awards-distinctions-events",
    "Conferences": "conferences",
    "Publications": "publications",
    "Selected Discography": "selected-discography",
    "Selected Projects & Production": "selected-projects-production",
    "Technology": "technology",
}

In [ ]:
section_ids = {
    "Education": "education",
    "Professional Experience": "professional-experience",
    "Teaching": "teaching",
    "Guest Lectures & Artist Talks": "guest-lectures-artist-talks",
    "Selected Awards, Distinctions, and Events": "selected-awards-distinctions-events",
    "Conferences": "conferences",
    "Publications": "publications",
    "Selected Discography": "selected-discography",
    "Selected Projects & Production": "selected-projects-production",
    "Technology": "technology",
}

In [ ]:
migrated_sections = []

for section in source_sections:
    section_heading = section.find(
        "h2",
        recursive=False,
    )

    section_title = section_heading.get_text(
        " ",
        strip=True,
    )

    groups = []

    for entry_list in section.find_all(
        "ul",
        recursive=False,
    ):
        group_heading = entry_list.find(
            "h3",
            recursive=False,
        )

        group_title = (
            group_heading.get_text(
                " ",
                strip=True,
            )
            if group_heading is not None
            else None
        )

        entries = [
            extract_entry(
                entry,
                section_title,
            )
            for entry in entry_list.find_all(
                "li",
                recursive=False,
            )
        ]

        groups.append(
            {
                "title": group_title,
                "entries": entries,
            }
        )

    migrated_sections.append(
        {
            "id": section_ids[
                section_title
            ],
            "title": section_title,
            "groups": groups,
        }
    )

migrated_cv = {
    "sections": migrated_sections,
}

In [ ]:
print("sections:", len(migrated_cv["sections"]))
print(
    "groups:",
    sum(
        len(section["groups"])
        for section in migrated_cv["sections"]
    ),
)
print(
    "entries:",
    sum(
        len(group["entries"])
        for section in migrated_cv["sections"]
        for group in section["groups"]
    ),
)

for section in migrated_cv["sections"]:
    print(
        section["id"],
        section["title"],
        [
            (
                group["title"],
                len(group["entries"]),
            )
            for group in section["groups"]
        ],
    )

In [ ]:
import json


migrated_cv_path = (
    project_root
    / "data"
    / "cv.migrated.json"
)

migrated_cv_path.write_text(
    json.dumps(
        migrated_cv,
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)

round_trip_cv = json.loads(
    migrated_cv_path.read_text(
        encoding="utf-8",
    )
)

print("saved to:", migrated_cv_path)
print(
    "round-trip match:",
    round_trip_cv == migrated_cv,
)

In [ ]:
all_entries = [
    entry
    for section in round_trip_cv["sections"]
    for group in section["groups"]
    for entry in group["entries"]
]

section_ids = [
    section["id"]
    for section in round_trip_cv["sections"]
]

boxed_entries = [
    entry
    for entry in all_entries
    if entry["type"] == "boxed"
]

link_entries = [
    entry
    for entry in all_entries
    if entry["type"] == "link"
]

plain_entries = [
    entry
    for entry in all_entries
    if entry["type"] == "plain"
]

planned_entries = [
    entry
    for entry in all_entries
    if entry.get("status") == "planned"
]

print("unique section IDs:", len(section_ids) == len(set(section_ids)))
print("boxed entries:", len(boxed_entries))
print("linked entries:", len(link_entries))
print("plain entries:", len(plain_entries))
print("planned entries:", len(planned_entries))
print(
    "all links have URLs:",
    all(entry["url"] for entry in link_entries),
)
print(
    "all labels are populated:",
    all(
        entry["label"]
        for entry in boxed_entries + link_entries
    ),
)
print(
    "all plain entries are populated:",
    all(entry["text"] for entry in plain_entries),
)

In [ ]:
section_id_by_title = {
    "Education": "education",
    "Professional Experience": "professional-experience",
    "Teaching": "teaching",
    "Guest Lectures & Artist Talks": "guest-lectures-artist-talks",
    "Selected Awards, Distinctions, and Events": "selected-awards-distinctions-events",
    "Conferences": "conferences",
    "Publications": "publications",
    "Selected Discography": "selected-discography",
    "Selected Projects & Production": "selected-projects-production",
    "Technology": "technology",
}

In [ ]:
migrated_sections = []

for section in source_sections:
    section_heading = section.find(
        "h2",
        recursive=False,
    )

    section_title = section_heading.get_text(
        " ",
        strip=True,
    )

    groups = []

    for entry_list in section.find_all(
        "ul",
        recursive=False,
    ):
        group_heading = entry_list.find(
            "h3",
            recursive=False,
        )

        group_title = (
            group_heading.get_text(
                " ",
                strip=True,
            )
            if group_heading is not None
            else None
        )

        entries = [
            extract_entry(
                entry,
                section_title,
            )
            for entry in entry_list.find_all(
                "li",
                recursive=False,
            )
        ]

        groups.append(
            {
                "title": group_title,
                "classes": entry_list.get(
                    "class",
                    [],
                ),
                "entries": entries,
            }
        )

    migrated_sections.append(
        {
            "id": section_id_by_title[
                section_title
            ],
            "title": section_title,
            "groups": groups,
        }
    )

migrated_cv = {
    "sections": migrated_sections,
}

In [ ]:
import json


migrated_cv_path = (
    project_root
    / "data"
    / "cv.migrated.json"
)

migrated_cv_path.write_text(
    json.dumps(
        migrated_cv,
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)

round_trip_cv = json.loads(
    migrated_cv_path.read_text(
        encoding="utf-8",
    )
)

print("saved to:", migrated_cv_path)
print(
    "round-trip match:",
    round_trip_cv == migrated_cv,
)

In [ ]:
for section in round_trip_cv["sections"]:
    for group in section["groups"]:
        if group["title"] is not None:
            print(
                section["title"],
                group["title"],
                group["classes"],
            )

In [ ]:
from pathlib import Path

from bs4 import BeautifulSoup
from jinja2 import (
    Environment,
    FileSystemLoader,
    select_autoescape,
)


project_root = Path.cwd().parent

template_directory = (
    project_root
    / "src"
    / "cv_caseyanderson"
    / "templates"
)

environment = Environment(
    loader=FileSystemLoader(
        template_directory
    ),
    autoescape=select_autoescape(
        enabled_extensions=(
            "html",
            "htm",
            "xml",
            "jinja",
        ),
        default_for_string=True,
        default=True,
    ),
    trim_blocks=True,
    lstrip_blocks=True,
)

template = environment.get_template(
    "index.html.jinja"
)

migrated_html = template.render(
    sections=round_trip_cv["sections"]
)

migrated_soup = BeautifulSoup(
    migrated_html,
    "html.parser",
)

print(
    "sections rendered:",
    len(
        migrated_soup.body.find_all(
            "div",
            recursive=False,
        )
    ),
)
print(
    "entries rendered:",
    len(
        migrated_soup.find_all("li")
    ),
)

In [ ]:
source_entry_texts = [
    entry.get_text(
        " ",
        strip=True,
    )
    for section in source_sections
    for entry in section.find_all("li")
]

rendered_entry_texts = [
    entry.get_text(
        " ",
        strip=True,
    )
    for entry in migrated_soup.find_all("li")
]

entry_mismatches = [
    {
        "index": index,
        "source": source_text,
        "rendered": rendered_text,
    }
    for index, (
        source_text,
        rendered_text,
    ) in enumerate(
        zip(
            source_entry_texts,
            rendered_entry_texts,
            strict=True,
        )
    )
    if source_text != rendered_text
]

print(
    "entry counts match:",
    len(source_entry_texts)
    == len(rendered_entry_texts),
)
print(
    "entry text preserved:",
    not entry_mismatches,
)
print("mismatches:", entry_mismatches)

In [ ]:
template = environment.get_template(
    "index.html.jinja"
)

migrated_html = template.render(
    sections=round_trip_cv["sections"]
)

migrated_soup = BeautifulSoup(
    migrated_html,
    "html.parser",
)

print(
    "sections rendered:",
    len(
        migrated_soup.body.find_all(
            "div",
            recursive=False,
        )
    ),
)
print(
    "entries rendered:",
    len(migrated_soup.find_all("li")),
)

In [ ]:
source_headings = [
    heading.get_text(
        " ",
        strip=True,
    )
    for heading in soup.find_all(
        ["h2", "h3"]
    )
]

rendered_headings = [
    heading.get_text(
        " ",
        strip=True,
    )
    for heading in migrated_soup.find_all(
        ["h2", "h3"]
    )
]

source_links = [
    {
        "text": link.get_text(
            " ",
            strip=True,
        ),
        "url": link.get("href"),
    }
    for link in soup.find_all("a")
]

rendered_links = [
    {
        "text": link.get_text(
            " ",
            strip=True,
        ),
        "url": link.get("href"),
    }
    for link in migrated_soup.find_all("a")
]

source_boxed_text = [
    span.get_text(
        " ",
        strip=True,
    )
    for span in soup.find_all(
        "span",
        class_="boxed",
    )
]

rendered_boxed_text = [
    span.get_text(
        " ",
        strip=True,
    )
    for span in migrated_soup.find_all(
        "span",
        class_="boxed",
    )
]

print(
    "headings preserved:",
    source_headings == rendered_headings,
)
print(
    "links preserved:",
    source_links == rendered_links,
)
print(
    "boxed labels preserved:",
    source_boxed_text == rendered_boxed_text,
)

In [ ]:
cv_path = (
    project_root
    / "data"
    / "cv.json"
)

cv_path.write_text(
    json.dumps(
        round_trip_cv,
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)

saved_cv = load_cv(cv_path)

print(
    "primary data matches migration:",
    saved_cv == round_trip_cv,
)

In [ ]:
from bs4 import BeautifulSoup

from cv_caseyanderson.content import load_cv
from cv_caseyanderson.render import render_cv


production_cv = load_cv(
    project_root / "data" / "cv.json"
)

production_html = render_cv(
    production_cv
)

production_soup = BeautifulSoup(
    production_html,
    "html.parser",
)

production_entry_texts = [
    entry.get_text(
        " ",
        strip=True,
    )
    for entry in production_soup.find_all("li")
]

print(
    "sections rendered:",
    len(
        production_soup.body.find_all(
            "div",
            recursive=False,
        )
    ),
)
print(
    "entries rendered:",
    len(production_entry_texts),
)
print(
    "source entry text preserved:",
    production_entry_texts
    == source_entry_texts,
)

In [ ]:
for section in migrated_cv["sections"]:
    for group in section["groups"]:
        for entry in group["entries"]:
            if entry["type"] == "plain":
                continue

            print(
                section["title"],
                repr(entry["prefix"]),
                repr(entry["label"]),
                repr(entry["suffix"]),
            )

In [ ]:
def extract_entry(entry, section_title):
    full_text = entry.get_text(
        " ",
        strip=True,
    )

    boxed = entry.find(
        "span",
        class_="boxed",
        recursive=False,
    )

    link = entry.find(
        "a",
        recursive=False,
    )

    if boxed is not None:
        label = boxed.get_text(
            " ",
            strip=True,
        )
        label_start = full_text.find(label)

        if label_start == -1:
            raise ValueError(
                "Boxed label was not found "
                "inside its entry"
            )

        text_before_label = full_text[
            :label_start
        ].strip()

        text_after_label = full_text[
            label_start + len(label):
        ].strip()

        is_planned = (
            section_title
            == "Selected Discography"
            and text_before_label == "("
            and text_after_label.endswith(")")
        )

        if is_planned:
            details_text = text_after_label[
                :-1
            ].strip()
        else:
            if text_before_label:
                raise ValueError(
                    "Unexpected text before "
                    "a boxed label"
                )

            details_text = text_after_label

        if details_text.startswith(","):
            details_text = details_text[
                1:
            ].strip()

        result = {
            "type": "boxed",
            "label": label,
            "details": details_text,
        }

        if is_planned:
            result.update(
                {
                    "status": "planned",
                    "prefix": "(",
                    "suffix": ")",
                }
            )

        return result

    if link is not None:
        label = link.get_text(
            " ",
            strip=True,
        )
        label_start = full_text.find(label)

        if label_start == -1:
            raise ValueError(
                "Linked label was not found "
                "inside its entry"
            )

        text_before_label = full_text[
            :label_start
        ].strip()

        if text_before_label:
            raise ValueError(
                "Unexpected text before "
                "a linked label"
            )

        details = full_text[
            label_start + len(label):
        ].strip()

        return {
            "type": "link",
            "label": label,
            "url": link.get("href"),
            "details": details,
        }

    return {
        "type": "plain",
        "text": full_text,
    }

In [ ]:
section_id_by_title = {
    "Education": "education",
    "Professional Experience": "professional-experience",
    "Teaching": "teaching",
    "Guest Lectures & Artist Talks": "guest-lectures-artist-talks",
    "Selected Awards, Distinctions, and Events": "selected-awards-distinctions-events",
    "Conferences": "conferences",
    "Publications": "publications",
    "Selected Discography": "selected-discography",
    "Selected Projects & Production": "selected-projects-production",
    "Technology": "technology",
}

migrated_sections = []

for section in source_sections:
    section_heading = section.find(
        "h2",
        recursive=False,
    )

    section_title = section_heading.get_text(
        " ",
        strip=True,
    )

    groups = []

    for entry_list in section.find_all(
        "ul",
        recursive=False,
    ):
        group_heading = entry_list.find(
            "h3",
            recursive=False,
        )

        group_title = (
            group_heading.get_text(
                " ",
                strip=True,
            )
            if group_heading is not None
            else None
        )

        entries = [
            extract_entry(
                entry,
                section_title,
            )
            for entry in entry_list.find_all(
                "li",
                recursive=False,
            )
        ]

        groups.append(
            {
                "title": group_title,
                "classes": entry_list.get(
                    "class",
                    [],
                ),
                "entries": entries,
            }
        )

    migrated_sections.append(
        {
            "id": section_id_by_title[
                section_title
            ],
            "title": section_title,
            "groups": groups,
        }
    )

migrated_cv = {
    "sections": migrated_sections,
}

print(
    "sections:",
    len(migrated_cv["sections"]),
)
print(
    "entries:",
    sum(
        len(group["entries"])
        for section in migrated_cv["sections"]
        for group in section["groups"]
    ),
)

In [ ]:
import json


migrated_cv_path = (
    project_root
    / "data"
    / "cv.migrated.json"
)

_ = migrated_cv_path.write_text(
    json.dumps(
        migrated_cv,
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)

round_trip_cv = json.loads(
    migrated_cv_path.read_text(
        encoding="utf-8",
    )
)

print(
    "round-trip match:",
    round_trip_cv == migrated_cv,
)

In [ ]:
planned_entries = [
    entry
    for section in round_trip_cv["sections"]
    for group in section["groups"]
    for entry in group["entries"]
    if entry.get("status") == "planned"
]

print(planned_entries)

In [ ]:
from cv_caseyanderson.render import render_cv


migrated_html = render_cv(
    round_trip_cv
)

migrated_soup = BeautifulSoup(
    migrated_html,
    "html.parser",
)

for planned_entry in planned_entries:
    label = planned_entry["label"]

    boxed_label = migrated_soup.find(
        "span",
        class_="boxed",
        string=label,
    )

    print(
        repr(
            boxed_label.parent.get_text()
        )
    )

In [ ]:
cv_path = (
    project_root
    / "data"
    / "cv.json"
)

_ = cv_path.write_text(
    json.dumps(
        round_trip_cv,
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)

saved_cv = load_cv(cv_path)

print(
    "primary data matches migration:",
    saved_cv == round_trip_cv,
)

In [ ]:
cv_path = (
    project_root
    / "data"
    / "cv.json"
)

cv_data = load_cv(cv_path)

_ = cv_path.write_text(
    json.dumps(
        cv_data,
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
    newline="\n",
)

print("normalized:", cv_path)